# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook loads and explores the FAIR^2 dataset of second primary colorectal cancer in cancer survivors using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined via a Croissant schema:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed. You may comment this if already installed.
!pip install mlcroissant --quiet

## 1. Data Loading

We will load the dataset and its metadata using `mlcroissant`, and print an overview.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Let's inspect which Record Sets are available, and the fields and columns within them using their `@id` values.

In [ ]:
# List all record sets (@ids)
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    # Each record set has an @id and likely .fields attribute
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', None)}")
    # List field @ids within the record set
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    Field @id: {field.id} | Name: {getattr(field, 'name', None)}")
            # Optional: print column @id if available
            if hasattr(field, 'columns'):
                for col in field.columns:
                    print(f"      Column @id: {col.id} | Name: {getattr(col, 'name', None)}")
    print("")

## 3. Data Extraction

Now, we'll load records from each available record set into Pandas DataFrames using their `@id`s.

In [ ]:
# We'll often have only one main record set in clinical tabular datasets
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        dataframes[record_set_id] = pd.DataFrame()

# For demonstration, show columns and preview for first record set
if record_set_ids:
    print(f"Loaded DataFrame for RecordSet @id: {record_set_ids[0]}")
    print("Columns:", dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)

Let's pick a numeric field, filter records, normalize, and group for summary statistics. All fields will be referenced by their `@id`.

In [ ]:
# ----
# You may want to pick appropriate `@id`s after running section 2 above; replace as needed:
# Example placeholders for field (@id)s:
#   numeric_field_id = '<field_id_for_age>'
#   group_field_id = '<field_id_for_sex>'
main_recordset = record_set_ids[0]
df = dataframes[main_recordset]

# Let's infer which column is numeric from recordset columns; we'll look for 'age' or a number variable
numeric_field_id = None
group_field_id = None

for col in df.columns:
    # Choose field id that sounds like age
    if "age" in col.lower():
        numeric_field_id = col
    # Choose a groupable field such as sex
    if group_field_id is None and ("sex" in col.lower() or "gender" in col.lower()):
        group_field_id = col

# Fallbacks in case not present
if numeric_field_id is None:
    numeric_cols = df.select_dtypes(include=["number"]).columns
    if len(numeric_cols) > 0:
        numeric_field_id = numeric_cols[0]

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

if numeric_field_id is not None and numeric_field_id in df.columns:
    # Remove non-numeric or missing values
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.10)  # 10th percentile as threshold, or choose fixed value
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a grouping field like 'sex' or other
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of our chosen numeric field, as well as its breakdown by a grouping variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='navy')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id], palette="Set2")
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

- We explored the dataset describing clinicopathological and molecular characteristics of second primary colorectal cancer in survivors.
- Using `mlcroissant`, we loaded metadata and records using only `@id` references throughout.
- We examined numeric variables (e.g. age), normalized and grouped them, and visualized their distributions.
- This enables further domain-specific statistical analysis or model development using this FAIR-format clinical dataset.